# Document Loader — PDFs & Transcripts
Uses LangChain `DirectoryLoader` to load all files from each folder automatically.

In [12]:
# pip install -qU langchain-groq
# %pip install chromadb
# %pip install fastapi uvicorn

In [ ]:
# !pip install langchain langchain-community langchain-openai pymupdf

In [14]:
# %pip install streamlit

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader,
    TextLoader,
)

In [ ]:
PDF_DIR        = "data"         # folder containing PDF books
TRANSCRIPT_DIR = "transcripts"  # folder containing .txt transcript files

## Load PDF Books

In [ ]:
pdf_loader = DirectoryLoader(
    path=PDF_DIR,
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
    use_multithreading=True,
)

pdf_docs = pdf_loader.load()
print(f"Total PDF pages loaded: {len(pdf_docs)}")

In [ ]:
# Inspect metadata — one entry per unique file
seen = set()
for doc in pdf_docs:
    src = doc.metadata.get("source")
    if src not in seen:
        seen.add(src)
        print(f"\n{'='*60}")
        print(f"File   : {src}")
        print(f"Title  : {doc.metadata.get('title', 'N/A')}")
        print(f"Author : {doc.metadata.get('author', 'N/A')}")
        print(f"Pages  : {doc.metadata.get('total_pages', 'N/A')}")
        print(f"Metadata: {doc.metadata}")

## Load Transcripts (.txt)

In [ ]:
transcript_loader = DirectoryLoader(
    path=TRANSCRIPT_DIR,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True,
)

transcript_docs = transcript_loader.load()
print(f"Total transcripts loaded: {len(transcript_docs)}")

In [ ]:
all_docs = pdf_docs + transcript_docs

print(f"PDF pages       : {len(pdf_docs)}")
print(f"Transcripts     : {len(transcript_docs)}")
print(f"Total documents : {len(all_docs)}")

In [ ]:
if pdf_docs:
    print("--- First PDF page ---")
    print(pdf_docs[0].page_content[:300])
    print(pdf_docs[0].metadata)

if transcript_docs:
    print("\n--- First transcript ---")
    print(transcript_docs[0].page_content[:300])
    print(transcript_docs[0].metadata)

## Chunking + Metadata Enrichment

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=500)
texts = text_splitter.split_documents(all_docs)
print(f"Total chunks: {len(texts)}")

### Metadata Schema (Pydantic)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ChunkMetadata(BaseModel):
    topics: list[str] = Field(
        description="Broad subject areas covered, max 3. e.g. 'vector databases', 'model training', 'retrieval augmented generation'"
    )
    concepts: list[str] = Field(
        description="Specific technical concepts discussed, max 4. e.g. 'semantic search', 'gradient descent', 'hybrid retrieval'"
    )
    difficulty: Literal["beginner", "intermediate", "advanced"] = Field(
        description="Technical difficulty level of this specific chunk"
    )
    content_type: Literal[
        "concept_explanation", "code_example", "best_practice",
        "case_study", "comparison", "tutorial", "note_or_warning"
    ] = Field(description="The type of content in this chunk")
    keywords: list[str] = Field(
        description="Important single terms, max 6. e.g. 'faiss', 'embedding', 'quantization'"
    )
    entities: list[str] = Field(
        description="Tools, frameworks, models, companies explicitly mentioned, max 6. e.g. 'LangChain', 'OpenAI', 'FAISS'"
    )

### LangChain Extractor Chain

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a metadata extractor for AI/ML technical content.
Extract structured metadata from the given text chunk.

IMPORTANT CONSTRAINTS - DO NOT DEVIATE:
- difficulty: MUST be one of: "beginner", "intermediate", or "advanced". If unclear, choose "intermediate".
- content_type: MUST be one of: "concept_explanation", "code_example", "best_practice", "case_study", "comparison", "tutorial", or "note_or_warning".
- topics: List max 3 items. Leave empty [] if not clearly AI/ML related.
- concepts: List max 4 items. Only include if clearly explained in the text.
- keywords: List max 6 items.
- entities: List max 6 items (tools, frameworks, models, companies).

Be precise. If content is metadata/administrative (e.g., copyright, author info), use:
- difficulty: "beginner"
- content_type: "note_or_warning"
- topics: []
- concepts: []
"""),
    ("human", "{text}")
])

# with_structured_output handles JSON mode + Pydantic parsing automatically
extractor = prompt | llm.with_structured_output(ChunkMetadata)

### Book-level Metadata (free — no LLM needed)

### Apply Metadata to All Chunks

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 50  # tune based on Groq rate limits

def generate_chunk_id(chunk_num: int) -> str:
    return f"chunk{chunk_num}"

def enrich_chunks_batch(chunks, doc_num=1):
    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Enriching batches"):
        batch = chunks[i : i + BATCH_SIZE]


        # Prepare inputs for all chunks in the batch
        inputs = [{"text": chunk.page_content} for chunk in batch]

        # Concurrent batch processing
        results = extractor.batch(
            inputs,
            config={"max_concurrency": BATCH_SIZE}
        )


        # Update metadata
        for batch_idx, (chunk, result) in enumerate(zip(batch, results), start=i + 1):

            # Stable readable chunk_id
            chunk_id = generate_chunk_id(
                chunk_num=batch_idx
            )

            # # Add chunk_id
            chunk.metadata["chunk_id"] = chunk_id
            # # Keep only useful metadata
            chunk.metadata["topics"] = result.topics
            chunk.metadata["concepts"] = result.concepts
            chunk.metadata["difficulty"] = result.difficulty
            chunk.metadata["content_type"] = result.content_type
            chunk.metadata["keywords"] = result.keywords
            chunk.metadata["entities"] = result.entities

    return chunks

chunks = enrich_chunks_batch(texts)
print("Done!")

In [ ]:
chunks[34].metadata

In [ ]:
print(texts[134].page_content)

In [ ]:
import json

chunks_json = []

for chunk in chunks:
    chunks_json.append({
        "page_content": chunk.page_content,
        "metadata": chunk.metadata
    })

with open("enriched_chunks.json", "w", encoding="utf-8") as f:
    json.dump(
        chunks_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Chunks saved to enriched_chunks.json")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}
encode_kwargs = {"normalize_embeddings": False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Use stable chunk_ids from metadata
chunk_ids = [doc.metadata["chunk_id"] for doc in texts]

vector_store.add_documents(
    documents=texts,
    ids=chunk_ids
)

In [ ]:
import faiss
from langchain_community.vectorstores import FAISS


vs = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)


In [ ]:
results = vs.similarity_search_with_score("what is agenticAI", k=5)

In [ ]:
retrieval_metadata = []
for rank, (doc, distance) in enumerate(results,start=1):
    similarity_score = 1 / (1 + float(distance))
    retrieval_metadata.append({
        "doc": doc.page_content,
        "faiss_rank": rank,
        "faiss_distance": float(distance),
        "faiss_score": similarity_score,
    })

In [ ]:
from sentence_transformers import CrossEncoder
question="what is agenticAI"
reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"
)
pairs  = [(question, doc.page_content) for doc, score in results]
scores = reranker.predict(pairs)

In [ ]:
for item, rerank_score in zip(retrieval_metadata,scores):
    item["rerank_score"] = float(rerank_score)

# Create a mapping from doc to its retrieval_metadata entry
doc_to_metadata = {id(results[i]): retrieval_metadata[i] for i in range(len(results))}

# Rerank and update with rerank_rank
reranked = sorted(zip(results, scores), key=lambda x: x[1], reverse=True)
for rerank_rank, (doc, rerank_score) in enumerate(reranked, start=1):
    metadata_entry = doc_to_metadata[id(doc)]
    metadata_entry["rerank_rank"] = rerank_rank

In [ ]:
retrieval_metadata

In [ ]:
vector_store.save_local("faiss_index")

In [ ]:
results

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"   # or "cpu"
)

In [ ]:
# Fields to exclude — PyMuPDF noise, not useful for LLM
EXCLUDE_FIELDS = {
    "producer", "creator", "creationdate", "moddate", "modDate",
    "creationDate", "trapped", "format", "file_path", "source",
    "subject", "total_pages"
}

def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata

        header = (
            f"[Source {i}]\n"
            f"Book      : {meta.get('title', meta.get('book_title', 'Unknown'))}\n"
            f"Author    : {meta.get('author', 'N/A')}\n"
            f"Chapter   : {meta.get('chapter_title', 'N/A')}\n"
            f"Page      : {meta.get('page', 'N/A')}\n"
            f"Topics    : {', '.join(meta.get('topics', []))}\n"
            f"Type      : {meta.get('content_type', 'N/A')}\n"
            f"Difficulty: {meta.get('difficulty', 'N/A')}"
        )
        formatted.append(f"{header}\n\nContent:\n{doc.page_content}")

    return "\n\n{'='*60}\n\n".join(formatted)

In [ ]:
def retrieval(question):
    results = vector_store.similarity_search(question, k=5)

    # 2. Rerank
    pairs  = [(question, doc.page_content) for doc in results]
    scores = reranker.predict(pairs)
    reranked = sorted(zip(results, scores), key=lambda x: x[1], reverse=True)
    # Extract documents from tuples: results is list of (doc, distance), scores are rerank scores
    top_docs = [(doc, score) for (doc, distance), score in reranked[:5]]

    # 3. Format
    context = format_docs(top_docs)
    return context

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os


def build_rag_chain():

    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,
        api_key=os.getenv("OPENAI_API_KEY"),
    )

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
You are an expert AI/ML teaching assistant with deep knowledge of
machine learning systems, LLM applications, MLOps, and generative AI.

Your job is to answer questions using ONLY the provided context retrieved from
authoritative AI/ML books and lecture transcripts.

Guidelines:
- Answer clearly and in a structured way
- Always cite your source using [Source N] inline for every point you make
- If the question has multiple parts, address each one separately
- If the context contains code examples, include and explain them
- If the answer is not present in the context, say:
  "I could not find relevant information in the provided sources."
- Do NOT use any prior knowledge outside the provided context
- Do NOT make up citations or references

At the end of your response, include a "Sources Used" section listing the
books and page numbers you referenced.
"""
        ),
        (
            "human",
            """
Context:
{context}

Question:
{question}

Answer:
"""
        )
    ])

    rag_chain = prompt | llm | StrOutputParser()

    return rag_chain


def generate_answer(question: str, context: str) -> str:
    rag_chain = build_rag_chain()

    response = rag_chain.invoke({
        "question": question,
        "context": context,
    })

    return response

In [ ]:
def retrieval_chain(question):
    results = vector_store.similarity_search(question, k=5)

    # 2. Rerank
    pairs  = [(question, doc.page_content) for doc in results]
    scores = reranker.predict(pairs)
    reranked = sorted(zip(results, scores), key=lambda x: x[1], reverse=True)
    # Extract documents from tuples: results is list of (doc, distance), scores are rerank scores
    top_docs = [(doc, score) for (doc, distance), score in reranked[:5]]

    # 3. Format
    context = format_docs(top_docs)
    response = generate_answer(question, context)
    return response, context

In [ ]:
question= "What is KV caching and how it is helpful in reducing latency?"
# output, context = retrieval_chain(question)

In [6]:
from main import run_query

run_query(
    question="What is the cross attention in transformer?",
    session_id="user_4"
)

2026-05-14 09:57:07,822 | INFO | main | ============================================================
2026-05-14 09:57:07,822 | INFO | main | Query: What is the cross attention in transformer?
2026-05-14 09:57:07,822 | INFO | main | ============================================================
2026-05-14 09:57:07,825 | INFO | retrieval.vectorstore | Loading FAISS index from './faiss_index' ...
2026-05-14 09:57:07,847 | INFO | retrieval.vectorstore | FAISS index loaded successfully
2026-05-14 09:57:07,847 | INFO | retrieval.reranker | Retrieving top 20 candidates ...
2026-05-14 09:57:07,898 | INFO | retrieval.reranker | Reranking 20 candidates ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.36it/s]


2026-05-14 09:57:08,494 | INFO | retrieval.reranker | Retrieval metadata: [{'chunk_id': 'chunk1986', 'faiss_rank': 1, 'faiss_distance': 0.8075720071792603, 'faiss_score': 0.5532283062739576, 'rerank_score': 0.030254730954766273, 'rerank_rank': 4}, {'chunk_id': 'chunk1956', 'faiss_rank': 2, 'faiss_distance': 0.8326529264450073, 'faiss_score': 0.5456570557196593, 'rerank_score': 0.011344903148710728, 'rerank_rank': 8}, {'chunk_id': 'chunk1968', 'faiss_rank': 3, 'faiss_distance': 0.8357484936714172, 'faiss_score': 0.5447369307110493, 'rerank_score': 0.0002835784980561584, 'rerank_rank': 12}, {'chunk_id': 'chunk847', 'faiss_rank': 4, 'faiss_distance': 0.874693751335144, 'faiss_score': 0.5334204582950185, 'rerank_score': 0.009260988794267178, 'rerank_rank': 9}, {'chunk_id': 'chunk1960', 'faiss_rank': 5, 'faiss_distance': 0.8966017365455627, 'faiss_score': 0.5272588233634029, 'rerank_score': 0.033479657024145126, 'rerank_rank': 3}, {'chunk_id': 'chunk1936', 'faiss_rank': 6, 'faiss_distance':

In [ ]:
run_query(
    question="How is it different from transformers?",
    session_id="user_1"
)

In [ ]:
run_query(
    question="How it is used in multi head attention?",
    session_id="user_1"
)

In [ ]:
print(context)

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("memory/chat_history.db")

df = pd.read_sql_query(
    "SELECT * FROM retrieval_traces",
    conn
)

display(df)

conn.close()

,id,session_id,query,chunk_id,document_title,page,faiss_rank,faiss_distance,faiss_score,rerank_rank,rerank_score,chunk_content,created_at
0,1,user_1,What is attention mechanism?,chunk1940,None,NaN,1,1.036651,0.491002,10,0.000040,in the sentence and I'm only looking at H's I'...,2026-05-14 04:25:53
1,2,user_1,What is attention mechanism?,chunk1970,None,NaN,2,1.067840,0.483596,15,0.000037,you could have these multiple attention heads ...,2026-05-14 04:25:53
2,3,user_1,What is attention mechanism?,chunk847,AI Engineering,84.0,3,1.091848,0.478046,1,0.682732,• Each value vector (V) represents the actual ...,2026-05-14 04:25:53
3,4,user_1,What is attention mechanism?,chunk1941,None,NaN,4,1.099730,0.476252,7,0.000051,refers refers to street so now when I'm Comput...,2026-05-14 04:25:53
4,5,user_1,What is attention mechanism?,chunk848,AI Engineering,85.0,5,1.101547,0.475840,2,0.150558,"8 Because input tokens are processed in batch,...",2026-05-14 04:25:53
5,6,user_1,What is attention mechanism?,chunk1924,None,NaN,6,1.109326,0.474085,11,0.000038,all these columns right so Alpha 1 one into th...,2026-05-14 04:25:53
6,7,user_1,What is attention mechanism?,chunk1923,None,NaN,7,1.111703,0.473551,3,0.030826,out and those can be my uh those are now with ...,2026-05-14 04:25:53
7,8,user_1,What is attention mechanism?,chunk1936,None,NaN,8,1.116796,0.472412,17,0.000037,[Music] so that brings us to the next module w...,2026-05-14 04:25:53
8,9,user_1,What is attention mechanism?,chunk1942,None,NaN,9,1.123770,0.470861,6,0.000072,in parallel as compared to to the previous cas...,2026-05-14 04:25:53
9,10,user_1,What is attention mechanism?,chunk1926,None,NaN,10,1.140351,0.467213,4,0.020590,for a given T it can be parallelized right and...,2026-05-14 04:25:53
